# Kiểm chứng số liệu Chương 4 trên kho CompanyX_DW
Notebook chỉ đọc. Mỗi mục đối chiếu một nhận định trong Chương 4 với dữ liệu thật trên `.\SQLEXPRESS`.

In [1]:
import os, re, warnings, pyodbc, pandas as pd
from IPython.display import display, HTML
warnings.filterwarnings('ignore', message='pandas only supports SQLAlchemy')
pd.set_option('display.max_columns', 30)
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..')) if os.path.basename(os.getcwd()) == 'kiem_chung' else os.getcwd()
def ket_noi():
    return pyodbc.connect('DRIVER={ODBC Driver 18 for SQL Server};SERVER=.\\SQLEXPRESS;DATABASE=CompanyX_DW;'
                          'Trusted_Connection=yes;TrustServerCertificate=yes', readonly=True)
cn = ket_noi()
def q(sql, *a):
    return pd.read_sql(sql, cn, params=a or None)
def chay_file(path):
    """Chạy một file .sql (tách theo GO) trên một kết nối riêng (bảng tạm # không lẫn giữa các file), trả về mọi bảng kết quả."""
    text = open(os.path.join(ROOT, path), encoding='utf-8-sig').read()
    out, cur = [], ket_noi().cursor()
    for batch in re.split(r'^\s*GO\s*$', text, flags=re.M | re.I):
        if not batch.strip(): continue
        cur.execute(batch)
        while True:
            if cur.description:
                out.append(pd.DataFrame.from_records(cur.fetchall(), columns=[c[0] for c in cur.description]))
            if not cur.nextset(): break
    return out
def bang(df, tieu_de):
    display(HTML(f'<div class="kc"><b>{tieu_de}</b>{df.to_html(index=False)}</div>'))

## A. Nhật ký lần chạy ETL cuối (mục 4.2.2, Hình 4.7)
Quy trình ETL ghi mỗi lần chạy vào `etl.RunLog` và từng bước vào `etl.StepLog`.

In [2]:
run = q("SELECT TOP 1 RunID, StartedAt, FinishedAt, Status FROM etl.RunLog ORDER BY StartedAt DESC")
steps = q("SELECT StepNo, StepName, SourceRows, LoadedRows, ExcludedRows, RejectedRows FROM etl.StepLog WHERE RunID=? ORDER BY StepNo", run.RunID[0])
bang(run, 'etl.RunLog – lần chạy cuối')
bang(steps, f'etl.StepLog – {len(steps)} bước của lần chạy cuối')

RunID,StartedAt,FinishedAt,Status
551BE3E7-7225-46D6-B06F-2593B2A3369E,2026-10-31 16:12:32,2026-10-31 16:12:34,SUCCESS


StepNo,StepName,SourceRows,LoadedRows,ExcludedRows,RejectedRows
1,Extract.Customer,19820,19820,0,0
2,Extract.Product,504,504,0,0
3,Extract.Territory,10,10,0,0
4,Extract.Order,31465,31465,0,0
5,Extract.OrderLine,121317,121317,0,0
6,Load.DimCustomer,19820,18484,1336,0
7,Load.DimProduct,504,504,0,0
8,Load.DimTerritory,10,10,0,0
9,Load.DimDate,1127,1127,0,0
10,Load.FactSalesOrder,31465,27382,4083,0


## B. 44 phép kiểm tra chất lượng sau ETL (mục 4.2.3, Bảng 4.3)
Chạy `sql/2_Cum2_ETL_moi/C2_03_Verify_and_Quality.sql` (chỉ ghi bảng tạm). Đếm số phép kiểm tra theo nhóm để đối chiếu câu "44 phép kiểm tra thuộc bảy nhóm".

In [3]:
kq = chay_file('sql/2_Cum2_ETL_moi/C2_03_Verify_and_Quality.sql')
chi_tiet = next(d for d in kq if {'GroupName', 'CheckName', 'FailedCount'} <= set(d.columns))
tong = chi_tiet.groupby('GroupName').agg(SoPhep=('CheckName', 'size'), Fail=('FailedCount', lambda s: int((s > 0).sum()))).reset_index()
bang(tong, f'C2_03: {len(chi_tiet)} phép kiểm tra, {int((chi_tiet.FailedCount > 0).sum())} FAIL')

GroupName,SoPhep,Fail
01. Số dòng,8,0
02. Mã nguồn,8,0
03. Khóa,6,0
04. Phạm vi/ánh xạ,8,0
05. Tiền,4,0
06. Rejection,6,0
07. Nhật ký,4,0


## C. 21 phép kiểm tra Customer Snapshot (mục 4.2.2.3, Hình 4.9)
Nhóm "Customer Snapshot và RFM" của Bảng 3.7 được kiểm riêng bằng `C3_03_Verify_Snapshot_Quality_READ_ONLY.sql`.

In [4]:
kq3 = chay_file('sql/3_Cum3A_Snapshot/C3_03_Verify_Snapshot_Quality_READ_ONLY.sql')
ct3 = next(d for d in kq3 if {'Category', 'CheckName', 'FailedRows'} <= set(d.columns))
tong3 = ct3.groupby('Category').agg(SoPhep=('CheckName', 'size'), Fail=('FailedRows', lambda s: int((s > 0).sum()))).reset_index()
bang(tong3, f'C3_03: {len(ct3)} phép kiểm tra, {int((ct3.FailedRows > 0).sum())} FAIL')

Category,SoPhep,Fail
01. Calendar,4,0
02. Coverage,4,0
03. Domain,4,0
04. RFM verification,5,0
05. Product verification,4,0


## D. Số bản ghi sau ETL (Bảng 4.1) và số khách theo mốc (Bảng 4.2)

In [5]:
dem = q("""SELECT 'DimCustomer' AS Bang, COUNT(*) AS SoDong FROM dbo.DimCustomer UNION ALL SELECT 'DimProduct', COUNT(*) FROM dbo.DimProduct
UNION ALL SELECT 'DimTerritory', COUNT(*) FROM dbo.DimTerritory UNION ALL SELECT 'DimDate', COUNT(*) FROM dbo.DimDate
UNION ALL SELECT 'FactSalesOrder', COUNT(*) FROM dbo.FactSalesOrder UNION ALL SELECT 'FactSalesOrderLine', COUNT(*) FROM dbo.FactSalesOrderLine""")
moc = q("SELECT CONVERT(CHAR(10), SnapshotDate, 103) AS Moc, COUNT(*) AS SoKhach FROM analytics.CustomerSnapshot GROUP BY SnapshotDate ORDER BY SnapshotDate")
bang(dem, 'Bảng 4.1 – số dòng sáu bảng Dimension và Fact')
bang(moc, f'Bảng 4.2 – {len(moc)} mốc trong analytics.CustomerSnapshot')

Bang,SoDong
DimCustomer,18484
DimProduct,504
DimTerritory,10
DimDate,1127
FactSalesOrder,27382
FactSalesOrderLine,59795


Moc,SoKhach
01/06/2013,5324
15/06/2013,5423
01/07/2013,5536
15/07/2013,5889
01/08/2013,6529
15/08/2013,7077
01/09/2013,7705
15/09/2013,8249
01/10/2013,8812
15/10/2013,9305


## E. Kiểm tra kết quả cụm 6 (mục 4.4.2, Hình 4.22)
Chạy `results/cum6/CUM6_04_Verify_READ_ONLY.sql`. Mong đợi: 18.360 khách có nhóm, 5 nhóm, 9 quy tắc (7 bật), 6.124 khuyến nghị, Z ≈ 256.502 USD.

In [6]:
kq6 = chay_file('results/cum6/CUM6_04_Verify_READ_ONLY.sql')
ten6 = ['Số khách có nhóm và số nhóm', 'Năm nhóm: số khách, RiskScore trung bình, số khách Cao', 'Kho quy tắc CareRule',
        'Danh sách chăm sóc theo kịch bản: số khách, chi phí, Z', 'Hồ sơ khách 19785 (màn hình 2)']
display(HTML('<div id="cum6">' + ''.join(f'<div class="kc"><b>{t}</b>{d.to_html(index=False)}</div>' for t, d in zip(ten6, kq6)) + '</div>'))